# Anima LoRA Trainer (SD3) for Google Colab

This notebook allows you to train LoRA models for Anima (based on SD3) using `sd-scripts`.
It handles environment setup (using `uv` and Python 3.12), directory configuration, and training execution.

In [ ]:
import os
import subprocess
import sys

# @markdown ### 1. Environment Setup
# @markdown This cell installs `uv`, clones the repository, and sets up the Python 3.12 environment with dependencies.
# @markdown Run this cell first and wait for it to complete.

# Install uv
print("Installing uv...")
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)

# Clone repository
repo_dir = "/content/sd-scripts"
if not os.path.exists(repo_dir):
    print("Cloning repository (sd3 branch)...")
    subprocess.run(["git", "clone", "-b", "sd3", "https://github.com/duongve13112002/sd-scripts", repo_dir], check=True)
else:
    print("Repository already exists.")

# Create venv with Python 3.12
venv_path = os.path.join(repo_dir, ".venv")
if not os.path.exists(venv_path):
    print("Creating virtual environment with Python 3.12...")
    subprocess.run(["uv", "venv", venv_path, "--python", "3.12"], check=True)

# Install dependencies
print("Installing dependencies...")
venv_python = os.path.join(venv_path, "bin", "python")

# Upgrade pip in venv
subprocess.run([venv_python, "-m", "pip", "install", "--upgrade", "pip"], check=True)

# Install torch, torchvision, xformers compatible with CUDA
print("Installing torch and xformers...")
subprocess.run([venv_python, "-m", "pip", "install", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run([venv_python, "-m", "pip", "install", "xformers", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

# Install requirements
print("Installing repository requirements...")
subprocess.run([venv_python, "-m", "pip", "install", "-r", os.path.join(repo_dir, "requirements.txt")], check=True)

# Uninstall rich
print("Uninstalling rich to prevent conflicts...")
subprocess.run([venv_python, "-m", "pip", "uninstall", "rich", "-y"], check=True)

# Install toml for config generation in this notebook (if not present in colab env, though usually is)
subprocess.run([sys.executable, "-m", "pip", "install", "toml"], check=True)

print("Environment setup complete.")

In [ ]:
import os
from google.colab import drive

# @markdown ### 2. Directory Configuration
# @markdown Mount Google Drive and set the paths for your dataset, output, and configuration files.

mount_drive = True # @param {type:"boolean"}
if mount_drive and not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

image_dir = "/content/drive/MyDrive/Lora/dataset" # @param {type:"string"}
output_dir = "/content/drive/MyDrive/Lora/output" # @param {type:"string"}
logging_dir = "/content/drive/MyDrive/Lora/logs" # @param {type:"string"}
config_dir = "/content/drive/MyDrive/Lora/config" # @param {type:"string"}
model_dir = "/content/drive/MyDrive/Lora/models" # @param {type:"string"}

# Create directories
for d in [image_dir, output_dir, logging_dir, config_dir, model_dir]:
    os.makedirs(d, exist_ok=True)

print(f"Directories configured:")
print(f"  Images: {image_dir}")
print(f"  Output: {output_dir}")
print(f"  Logs:   {logging_dir}")
print(f"  Config: {config_dir}")
print(f"  Models: {model_dir}")

In [ ]:
import toml
import os

# @markdown ### 3. Training Configuration (SD3 / Anima)
# @markdown Configure the hyperparameters for your LoRA training.

# @markdown #### Project Settings
project_name = "anima_lora_v1" # @param {type:"string"}

# @markdown #### Model Paths
dit_path = "/content/drive/MyDrive/Lora/models/anima_preview.safetensors" # @param {type:"string"}
qwen3_path = "/content/drive/MyDrive/Lora/models/qwen3_06b" # @param {type:"string"}
vae_path = "/content/drive/MyDrive/Lora/models/wan_vae.safetensors" # @param {type:"string"}
t5_tokenizer_path = "" # @param {type:"string"}

# @markdown #### Network Configuration
network_dim = 16 # @param {type:"integer"}
network_alpha = 8 # @param {type:"integer"}
network_module = "networks.lora_anima" # @param {type:"string"}
train_unet_only = False # @param {type:"boolean"}
train_text_encoder = False # @param {type:"boolean"}

# @markdown #### Training Hyperparameters
learning_rate = 1e-4 # @param {type:"number"}
unet_lr = 1e-4 # @param {type:"number"}
text_encoder_lr = 5e-5 # @param {type:"number"}
lr_scheduler = "constant" # @param ["constant", "cosine", "cosine_with_restarts", "polynomial", "linear", "constant_with_warmup"]
lr_warmup_steps = 0 # @param {type:"integer"}
optimizer_type = "AdamW8bit" # @param ["AdamW8bit", "AdamW", "Adafactor", "Prodigy"]
max_train_epochs = 10 # @param {type:"integer"}
save_every_n_epochs = 1 # @param {type:"integer"}
train_batch_size = 1 # @param {type:"integer"}
mixed_precision = "bf16" # @param ["no", "fp16", "bf16"]
save_precision = "bf16" # @param ["float", "fp16", "bf16"]
gradient_checkpointing = True # @param {type:"boolean"}
seed = 42 # @param {type:"integer"}

# @markdown #### Anima Specific
timestep_sample_method = "logit_normal" # @param ["logit_normal", "uniform"]
discrete_flow_shift = 3.0 # @param {type:"number"}
sigmoid_scale = 1.0 # @param {type:"number"}
weighting_scheme = "uniform" # @param ["uniform", "sigma_sqrt", "cosmap", "none"]

# @markdown #### Dataset Configuration
resolution = 1024 # @param {type:"integer"}
num_repeats = 10 # @param {type:"integer"}
shuffle_caption = True # @param {type:"boolean"}

# Generate Dataset Config
dataset_config = {
    "general": {
        "shuffle_caption": shuffle_caption,
        "keep_tokens": 1,
        "bucket_reso_steps": 64,
        "enable_bucket": True
    },
    "datasets": [
        {
            "resolution": resolution,
            "batch_size": train_batch_size,
            "subsets": [
                {
                    "image_dir": image_dir,
                    "num_repeats": num_repeats
                }
            ]
        }
    ]
}

dataset_config_path = os.path.join(config_dir, "dataset_config.toml")
with open(dataset_config_path, "w") as f:
    toml.dump(dataset_config, f)
print(f"Dataset config saved to {dataset_config_path}")

# Generate Training Config (Flat TOML for argparse)
config_args = {
    # Model
    "dit_path": dit_path,
    "qwen3_path": qwen3_path,
    "vae_path": vae_path,
    
    # Network
    "network_module": network_module,
    "network_dim": network_dim,
    "network_alpha": network_alpha,
    "network_train_unet_only": train_unet_only,

    # Optimizer & LR
    "learning_rate": learning_rate,
    "lr_scheduler": lr_scheduler,
    "lr_warmup_steps": lr_warmup_steps,
    "optimizer_type": optimizer_type,
    "unet_lr": unet_lr,
    "text_encoder_lr": text_encoder_lr,

    # General Training
    "output_dir": output_dir,
    "output_name": project_name,
    "dataset_config": dataset_config_path,
    "max_train_epochs": max_train_epochs,
    "save_every_n_epochs": save_every_n_epochs,
    "mixed_precision": mixed_precision,
    "save_precision": save_precision,
    "gradient_checkpointing": gradient_checkpointing,
    "seed": seed,
    "xformers": True,
    "sdpa": False,
    "cache_latents": True,
    "cache_text_encoder_outputs": True,
    "logging_dir": logging_dir,

    # Anima Specific
    "timestep_sample_method": timestep_sample_method,
    "discrete_flow_shift": discrete_flow_shift,
    "sigmoid_scale": sigmoid_scale,
    "weighting_scheme": weighting_scheme,
}

if t5_tokenizer_path:
    config_args["t5_tokenizer_path"] = t5_tokenizer_path

if mixed_precision == "bf16":
    config_args["full_bf16"] = True
elif mixed_precision == "fp16":
    config_args["full_fp16"] = True

config_path = os.path.join(config_dir, "config.toml")
with open(config_path, "w") as f:
    toml.dump(config_args, f)

print(f"Training config saved to {config_path}")

In [ ]:
# @markdown ### 4. Start Training
# @markdown This cell runs the training command using the generated configuration.

accelerate_path = "/content/sd-scripts/.venv/bin/accelerate"
script_path = "/content/sd-scripts/anima_train_network.py"

print("Starting training...")
print(f"Using config: {config_path}")

# Construct command for display
cmd = f"{accelerate_path} launch --num_cpu_threads_per_process 1 {script_path} --config_file {config_path}"
print(f"Executing: {cmd}")

# Run command
!{accelerate_path} launch --num_cpu_threads_per_process 1 {script_path} --config_file "{config_path}"